# Klein VAE Experiments

In [ ]:
%load_ext autoreload
%autoreload 2

import rootutils
root = rootutils.setup_root(search_from=".", indicator=".project-root", pythonpath=True, cwd=True)

In [ ]:
from src.data.uniform_filters_datamodule import UniformFiltersDataModule
from src.models.klein_vae_module import KleinVAEModule
from src.models.components.vae import SimpleVAE

from src.data.components.utils import generate_klein_filter_matrix
from src.utils.topology_utils import project_to_klein

import matplotlib.pyplot as plt

from lightning.pytorch.trainer import Trainer
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
import torch

import random
import numpy as np
import pandas as pd

from ripser import ripser
from persim import plot_diagrams


In [ ]:
BATCH_SIZE = 16

In [ ]:
datamodule = UniformFiltersDataModule(num_samples_per_angle=100, batch_size=BATCH_SIZE)
datamodule.setup()
filter_example = datamodule.train_dataloader().dataset[0][0].reshape(3,3)

train_loader = datamodule.train_dataloader()
total_examples = len(train_loader.dataset)
print(f"Total training examples: {total_examples}")

plt.imshow(filter_example, cmap='gray')
plt.colorbar()
plt.title('Filter Example')
plt.show()

### Testing that samples from `UniformFiltersDataModule` lie on the Klein bottle

In [ ]:
dataset = datamodule.train_dataloader().dataset
indices = random.sample(range(len(dataset)), 500)
batch = np.array([dataset[i][0].numpy() for i in indices])

In [ ]:
diagrams2 = ripser(batch, maxdim=2, coeff=2)['dgms']
diagrams3 = ripser(batch, maxdim=2, coeff=3)['dgms']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('$\mathbb{Z}_2$')
plot_diagrams(diagrams2, ax=axes[0], show=False)

axes[1].set_title('$\mathbb{Z}_3$')
plot_diagrams(diagrams3, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

### Running the model

In [ ]:
vae_model = SimpleVAE(input_dim=3**2, hidden_dims=[32, 32, 32, 32, 32], latent_dim=2)
klein_vae_module = KleinVAEModule(
    model=vae_model,
    optimizer=torch.optim.Adam,
    recon_loss=torch.nn.BCELoss(reduction="mean"),
)


def sample_plane_latents(module, num_samples, prior_mean=None, prior_std=None):
    latent_dim = module.model.latent_dim
    device = next(module.parameters()).device
    mean = module.prior_mean if prior_mean is None else prior_mean
    std = module.prior_std if prior_std is None else prior_std
    return mean + std * torch.randn(num_samples, latent_dim, device=device)


def sample_klein_latents(module, num_samples, return_unprojected=False):
    z_plane = sample_plane_latents(module, num_samples)
    z_klein = project_to_klein(z_plane)
    if return_unprojected:
        return z_klein, z_plane
    return z_klein


def encode_mu(module, x, project=False):
    if x.ndim == 1:
        x = x.unsqueeze(0)
    x = x.to(next(module.parameters()).device)
    mu, log_var = module.model.encode(x)
    if project:
        mu = project_to_klein(mu)
    return mu, log_var


def decode_latents(module, z):
    return module.model.decode(z)


#### Checking the filters BEFORE training

In [ ]:
samples = sample_klein_latents(klein_vae_module, num_samples=10)
filters_from_decoder = decode_latents(klein_vae_module, samples).reshape(-1, 3, 3)

plt.figure(figsize=(10, 5))
for i, filter_img in enumerate(filters_from_decoder):
    plt.subplot(2, 5, i + 1)
    plt.imshow(filter_img.detach().cpu().numpy(), cmap='gray')
    plt.axis('off')
    plt.title(f'Sample {i+1}')
plt.tight_layout()
plt.show()

In [ ]:
samples = sample_klein_latents(klein_vae_module, num_samples=500)
decoded = decode_latents(klein_vae_module, samples).detach().cpu().numpy()
batch = decoded

diagrams = ripser(batch, maxdim=2)['dgms']
plot_diagrams(diagrams, show=True, title='Persistence Diagrams for Decoded Filters')

### Training

In [ ]:
klein_vae_module.kl_weight = 1e-3

In [ ]:
trainer = Trainer(accelerator="cpu", max_epochs=100, enable_progress_bar=True)
trainer.fit(klein_vae_module, datamodule=datamodule)

In [ ]:
# Retrieve and display the logged metrics after training
metrics = trainer.callback_metrics
print(metrics)

In [ ]:
samples = sample_klein_latents(klein_vae_module, num_samples=20)
filters_from_decoder = decode_latents(klein_vae_module, samples).reshape(-1, 3, 3)


plt.figure(figsize=(30, 15))
for i, filter_img in enumerate(filters_from_decoder):
    plt.subplot(4, 5, i + 1)
    plt.imshow(filter_img.detach().cpu().numpy(), cmap='gray')
    plt.axis('off')
    plt.title(f'{samples[i].detach().cpu().numpy()}')
plt.tight_layout()
plt.show()

### Testing that decoded samples also lie on the Klein bottle

In [ ]:
samples = sample_klein_latents(klein_vae_module, num_samples=500)
decoded = decode_latents(klein_vae_module, samples)
batch = decoded.detach().cpu().numpy()

In [ ]:
diagrams2 = ripser(batch, maxdim=2, coeff=2)['dgms']
diagrams3 = ripser(batch, maxdim=2, coeff=3)['dgms']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('$\mathbb{Z}_2$')
plot_diagrams(diagrams2, ax=axes[0], show=False)

axes[1].set_title('$\mathbb{Z}_3$')
plot_diagrams(diagrams3, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

### Try to find the problem

In [ ]:
n_points = 20

grid_on_flat_klein = torch.cartesian_prod(torch.linspace(0, 1, n_points), 
                                          torch.linspace(0, 1, n_points))
filters = torch.stack([generate_klein_filter_matrix(theta_1 * 2 * torch.pi, theta_2 * 2 * torch.pi, size=3, midpoint=False).flatten() 
                        for theta_1, theta_2 in grid_on_flat_klein])

In [ ]:
mu_on_klein, log_var = encode_mu(klein_vae_module, filters, project=True)
sigma_on_klein = log_var.exp().sqrt()

In [ ]:
# scatter plot of initial grid coordinates vs. projected μ on the Klein bottle
initial_coords = grid_on_flat_klein.cpu().numpy()
klein_coords = mu_on_klein.detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

#axes[0].scatter(datamodule.coordinates[:, 0], datamodule.coordinates[:, 1], c='green', alpha=0.5, label='Uniform Samples')
#axes[0].legend()

axes[0].scatter(initial_coords[:, 0], initial_coords[:, 1], c='red', alpha=np.linspace(0.2,0.99,n_points**2))
axes[0].set_xlabel(r'$\theta_1$')
axes[0].set_ylabel(r'$\theta_2$')
axes[0].set_title('Initial Flat Torus Coordinates')

axes[1].scatter(klein_coords[:, 0], klein_coords[:, 1], c='blue', alpha=np.linspace(0.2,0.99,n_points**2))

axes[1].set_xlabel(r'$\theta_1$')
axes[1].set_ylabel(r'$\theta_2$')
axes[1].set_title('Projected μ')

plt.tight_layout()
plt.show()

127, 101, 256, 107

In [ ]:
random_idx = np.random.randint(0, len(grid_on_flat_klein), size=10)

for i in random_idx:
    filter_example = generate_klein_filter_matrix(grid_on_flat_klein[i][0], grid_on_flat_klein[i][1])
    encoded_coords, _ = encode_mu(klein_vae_module, filter_example.flatten(), project=True)
    reconstructed_filter = decode_latents(klein_vae_module, encoded_coords).reshape(3, 3)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(filter_example, cmap='gray')
    axes[1].imshow(reconstructed_filter.detach().cpu().numpy(), cmap='gray')
    axes[0].set_title(f'Original Filter Example: {i}')
    axes[1].set_title(f'Encoded Filter Example: {i}')
    plt.tight_layout()
    # plt.show()

In [ ]:
filter_example, encoded_filter.reshape(3,3)

In [ ]:
samples_on_klein, samples_from_model = sample_klein_latents(
    klein_vae_module,
    num_samples=100,
    return_unprojected=True,
)


plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(mu_on_klein[:, 0].detach().cpu().numpy(), mu_on_klein[:, 1].detach().cpu().numpy(), c='blue', label='Encoded Mean')
plt.scatter(samples_on_klein[:, 0].detach().cpu().numpy(), samples_on_klein[:, 1].detach().cpu().numpy(), c='red', label='Sampled Points')
plt.title('Samples on Klein Bottle')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(grid_on_flat_klein[:, 0].detach().cpu().numpy(), grid_on_flat_klein[:, 1].detach().cpu().numpy(), c='blue', label='Original Grid')
plt.scatter(samples_from_model[:, 0].detach().cpu().numpy(), samples_from_model[:, 1].detach().cpu().numpy(), c='red', label='Unprojected Samples')
plt.title('Samples on the Flat Klein Domain')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()

plt.tight_layout()
plt.show()

## VIsualising logs

In [ ]:
logs_dir = trainer.logger.log_dir

logs_df = pd.read_csv(f"{logs_dir}/metrics.csv")

plt.plot(logs_df['recon_loss'].dropna())

In [ ]:
#trainer.save_checkpoint("MODELS/16-32-64-checkpoint.ckpt")

## Circles Dataset

In [ ]:
from src.data.circles_datamodule import CirclesDatamodule

In [ ]:
IMG_SIZE = 30
BATCH_SIZE = 1024
datamodule = CirclesDatamodule(num_images=100000, image_linear_pixel_size=IMG_SIZE, batch_size=BATCH_SIZE)
datamodule.setup()

In [ ]:
filter_example = datamodule.train_dataloader().dataset[100][0].reshape(IMG_SIZE,IMG_SIZE)

train_loader = datamodule.train_dataloader()
total_examples = len(train_loader.dataset)
print(f"Total training examples: {total_examples}")

plt.imshow(filter_example, cmap='gray')
plt.colorbar()
plt.title('Filter Example')
plt.show()

In [ ]:
vae_model = SimpleVAE(input_dim=IMG_SIZE**2, hidden_dims=[1024, 512, 128, 32], latent_dim=2)
klein_vae_module = KleinVAEModule(
    model=vae_model,
    optimizer=torch.optim.Adam,
    recon_loss=torch.nn.BCELoss(reduction="mean"),
    lr=1e-3,
    kl_weight=1e-1,
    prior_mean=0.5,
    prior_std=0.1,
)

In [ ]:
trainer = Trainer(accelerator="cpu", max_epochs=50, enable_progress_bar=True)#, callbacks=[EarlyStopping(monitor="recon_loss", patience=5, mode="min")])
trainer.fit(klein_vae_module, datamodule=datamodule)

In [ ]:
logs_df

In [ ]:
logs_dir = trainer.logger.log_dir

logs_df = pd.read_csv(f"{logs_dir}/metrics.csv")

plt.plot(logs_df['train_recon_loss_epoch'].dropna().reset_index(drop=True), label='Reconstruction Loss')
plt.plot(logs_df['train_kl_loss_epoch'].dropna().reset_index(drop=True), label='KL Divergence')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Losses Over Epochs')
plt.legend()
plt.show()

In [ ]:
random_idx = np.random.randint(0, len(datamodule.val_dataloader().dataset), size=10)

for i in random_idx:
    image = datamodule.val_dataloader().dataset[i][0]

    proj_rec, _ = encode_mu(klein_vae_module, image, project=True)
    print(proj_rec)
    rec = decode_latents(klein_vae_module, proj_rec).reshape(IMG_SIZE, IMG_SIZE)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(image.reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    axes[1].imshow(rec.detach().cpu().numpy(), cmap='gray')
    axes[0].set_title(f'Original Filter Example: {i}')
    axes[1].set_title(f'Encoded Filter Example: {i}')
    plt.tight_layout()
    # plt.show()

In [ ]:
#trainer.save_checkpoint('MODELS/HOORAY_CIRCLES.ckpt')

In [ ]:
val_points_on_klein = torch.cat(
    [
        encode_mu(klein_vae_module, datamodule.val_dataloader().dataset[i][0], project=True)[0]
        for i in range(len(datamodule.val_dataloader().dataset))
    ],
    dim=0,
).detach().cpu().numpy()

In [ ]:
plt.scatter(val_points_on_klein[:,0], val_points_on_klein[:,1])

### Homology

In [ ]:
dataset = datamodule.val_dataloader().dataset
indices = random.sample(range(len(dataset)), 500)
batch = np.array([dataset[i][0].numpy() for i in indices])

In [ ]:
diagrams2 = ripser(batch, maxdim=2, coeff=2)['dgms']
diagrams3 = ripser(batch, maxdim=2, coeff=3)['dgms']

In [ ]:
from persim import bottleneck

In [ ]:
bottleneck(diagrams2[1], diagrams3[1])

In [ ]:
# load the saved persistence diagrams
npzfile = np.load("diagrams_Z3.npz")
h0 = npzfile["h0"]
h1 = npzfile["h1"]
h2 = npzfile["h2"]

# optionally inspect
print(f"h0 shape: {h0.shape}, h1 shape: {h1.shape}, h2 shape: {h2.shape}")

In [ ]:
"Reconstructed PD over $\mathbb{Z}_2$ at Epoch " + str(3)

In [ ]:
from src.utils.topology_utils import compute_persistence_diagrams

res = compute_persistence_diagrams(batch, maxdim=2, coeffs=[2,3])

In [ ]:
for k, v in res.items():
    print(f"Coefficient: {k}")
    for dim, diagram in enumerate(v):
        print(f"  Dimension {dim}: {diagram.shape} points")

In [ ]:
np.savez_compressed("diagrams_Z2.npz", *diagrams2)

In [ ]:
np.savez_compressed("test.npz", diagrams2)

In [ ]:
from src.utils.topology_utils import plot_persistence_diagram_detailed

In [ ]:
fig, (ax_z2, ax_z3) = plt.subplots(1, 2, figsize=(16, 10))

# 2. Call the reusable function for the Z2 diagram
plot_persistence_diagram_detailed(
    diagram=diagrams2, 
    title=r'$\mathbb{Z}_2$', 
    ax=ax_z2,
    show_ylabels=True  # Show y-axis for the first plot
)

plot_persistence_diagram_detailed(
    diagram=diagrams3, 
    title=r'$\mathbb{Z}_3$', 
    ax=ax_z3,
    show_ylabels=False # Hide y-axis for the second plot
)

fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('$\mathbb{Z}_2$')
plot_diagrams(diagrams2, ax=axes[0], show=False)

axes[1].set_title('$\mathbb{Z}_3$')
plot_diagrams(diagrams3, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

### Grid

In [ ]:
#num_samples_per_angle = self.num_samples_per_angle
image_linear_pixel_size = IMG_SIZE
num_images = 5
circle_radius = 0.3

xs = torch.linspace(0,1,image_linear_pixel_size)
ys = torch.linspace(0,2,2*image_linear_pixel_size)

grid_x, grid_y = torch.meshgrid(xs, ys, indexing='ij')

imagess = []

imagess_flat = []

for i in range(num_images+1):
    images = []
    for j in range(num_images+1):
        # circle center
        cc_x = 1+(1.*i/(num_images-1))#1+2*torch.rand(1)
        cc_y = 2+1.*j/(num_images-1)#2+torch.rand(1)
        # 0,0
        gx, gy = grid_x, grid_y
        c00 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 0,1
        gx, gy = grid_x, grid_y+2
        c01 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 0,2
        gx, gy = grid_x, grid_y+4
        c02 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 1,0
        gx, gy = grid_x+1, grid_y
        c10 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 1,1
        gx, gy = grid_x+1, grid_y+2
        c11 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 1,2
        gx, gy = grid_x+1, grid_y+4
        c12 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 2,0
        gx, gy = grid_x+2, grid_y
        c20 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 2,1
        gx, gy = grid_x+2, grid_y+2
        c21 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)
        # 2,2
        gx, gy = grid_x+2, grid_y+4
        c22 = ((gx-cc_x)**2+(gy-cc_y)**2 < circle_radius**2)

        torus = (c00+c01+c02+c10+c11+c12+c20+c21+c22)

        klein = torus[:,:image_linear_pixel_size] + torch.flip(torus[:,image_linear_pixel_size:],dims=(0,))
        
        images.append(klein)
        
        imagess_flat.append(klein.flatten())
        
    imagess.append(images)
    
tensor = torch.stack(imagess_flat)

In [ ]:
n = num_images

fig = plt.figure(figsize=(10,10))
ax = fig.subplots(n,n)

for i in range(0,n):
    for j in range(0,n):
        ax[i,j].imshow(imagess[i][j], 
                        cmap='gray')
        ax[i,j].axes.get_xaxis().set_ticks([])
        ax[i,j].axes.get_yaxis().set_ticks([])

In [ ]:
imagess = np.load("true_grid_images.npy")

### True Homology

In [ ]:
batch = np.array([imagess[i][j].numpy().flatten() for i in range(len(imagess)) for j in range(len(imagess[i]))])
batch.shape

In [ ]:
diagrams2 = ripser(batch, maxdim=2, coeff=2)['dgms']
diagrams3 = ripser(batch, maxdim=2, coeff=3)['dgms']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('$\mathbb{Z}_2$')
plot_diagrams(diagrams2, ax=axes[0], show=False)

axes[1].set_title('$\mathbb{Z}_3$')
plot_diagrams(diagrams3, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

### Decoded grid values

In [ ]:
encoded_values, _ = encode_mu(klein_vae_module, tensor.to(torch.float), project=True)
decoded_images = decode_latents(klein_vae_module, encoded_values)
decoded_images_square = decoded_images.reshape(num_images+1, num_images+1, image_linear_pixel_size, image_linear_pixel_size).detach().cpu().numpy()
decoded_images_np = decoded_images.detach().cpu().numpy()

In [ ]:
np.save("decoded_images_square.npy", decoded_images_square)

In [ ]:
n = num_images

fig = plt.figure(figsize=(10,10))
ax = fig.subplots(n,n)

for i in range(0,n):
    for j in range(0,n):
        ax[i,j].imshow(decoded_images_square[i][j], 
                        cmap='gray')
        ax[i,j].axes.get_xaxis().set_ticks([])
        ax[i,j].axes.get_yaxis().set_ticks([])

In [ ]:
diagrams2 = ripser(decoded_images_np, maxdim=2, coeff=2)['dgms']
diagrams3 = ripser(decoded_images_np, maxdim=2, coeff=3)['dgms']

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].set_title('Reconstructed Images over $\mathbb{Z}_2$')
plot_diagrams(diagrams2, ax=axes[0], show=False)

axes[1].set_title('Reconstructed Images over $\mathbb{Z}_3$')
plot_diagrams(diagrams3, ax=axes[1], show=False)

plt.tight_layout()
plt.show()

In [ ]:
trainer.save_checkpoint("MODELS/klein-vae-circles-best.ckpt")